In [11]:
import numpy as np
import pandas as pd
import os
import cv2
from fastai.vision import *

In [39]:
print(" * Data csv loading ...")
df_movies = pd.read_csv("dataset/IMDB_movies.csv", sep=",")
size_movies_df = len(df_movies.index)
print("Base dataframe has", size_movies_df, "rows.")
print(" * Training csv loading ...")
df_train = pd.read_csv("dataset/Multi_Label_dataset/train.csv")
size_train_df = len(df_movies.index)
print("Base dataframe has", size_train_df, "rows.")
print("Create the different column in Data csv to get AI results ...")
df_movies['Test_AI'] = pd.Series([float('nan')]*len(df_movies))
print("Test_AI column created in df_movies")


In [6]:
id_to_genre = {}
for i in range(len(df_train)):
    id_to_genre[df_train.loc[i, "Id"]] = df_train.loc[i, "Genre"].split("|")

In [7]:
def preprocess_poster(poster_path):
    poster = cv2.imread(poster_path)
    poster = cv2.cvtColor(poster, cv2.COLOR_BGR2RGB)
    poster = cv2.resize(poster, (224, 224))
    poster = poster.astype(np.float32) / 255.0
    poster = np.expand_dims(poster, axis=0)
    return poster

In [19]:
posters = {}
folder_path = "dataset/Multi_Label_dataset/images/"
for filename in os.listdir(folder_path):
    if filename.endswith(".jpg"):
        poster_path = os.path.join(folder_path, filename)
        poster_id = filename[:-4]
        posters[poster_id] = preprocess_poster(poster_path)

In [ ]:
df_movies["poster"] = df_movies["imdb_title_id"].map(posters)


In [ ]:
df_movies.dropna(subset=["poster"], inplace=True)


In [ ]:
posters = {}

folder_path = "dataset/Multi_Label_dataset/images/"
for filename in os.listdir(folder_path):
    poster_path = os.path.join(folder_path, filename)
    poster_id = filename[:-4]
    posters[poster_id] = preprocess_poster(poster_path)

# Check for duplicate images
hashes = {}
duplicates = []
for folder_name in os.listdir(images_path):
    folder_path = os.path.join(images_path, folder_name)
    for filename in os.listdir(folder_path):
        if filename.endswith(".jpg"):
            poster_path = os.path.join(folder_path, filename)
            with open(poster_path, 'rb') as f:
                hash_value = hashlib.md5(f.read()).hexdigest()
            if hash_value in hashes:
                duplicates.append((filename, hashes[hash_value]))
            else:
                hashes[hash_value] = filename

if len(duplicates) > 0:
    print("Duplicate images found:")
    for pair in duplicates:
        print(pair)

# Check for missing labels
metadata = {}
with open(metadata_path, 'r') as f:
    for line in f:
        values = line.strip().split(',')
        poster_id = values[0]
        labels = values[1:]
        metadata[poster_id] = labels

missing_labels = []
for poster_id in hashes.values():
    if poster_id not in metadata:
        missing_labels.append(poster_id)

if len(missing_labels) > 0:
    print("Images with missing labels found:")
    for poster_id in missing_labels:
        print(poster_id)

# Remove corrupted images
corrupted = []
for folder_name in os.listdir(images_path):
    folder_path = os.path.join(images_path, folder_name)
    for filename in os.listdir(folder_path):
        if filename.endswith(".jpg"):
            poster_path = os.path.join(folder_path, filename)
            try:
                with Image.open(poster_path) as img:
                    img.load()
            except:
                corrupted.append((poster_path))
                os.remove(poster_path)

if len(corrupted) > 0:
    print("Corrupted images found:")
    for poster_path in corrupted:
        print(poster_path)


In [36]:
import numpy as np
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from PIL import Image
from tensorflow.keras.preprocessing import image

def extract_features(model, img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    img = np.squeeze(img, axis=0) # remove the extra dimension
    feat = model.predict(np.expand_dims(img, axis=0)).ravel()
    return feat

# Load the pre-trained ResNet50 model
model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Extract features from each movie poster image
features = {}
for poster_id, poster in posters.items():
    # Preprocess the image
    img = extract_features(model, poster)
    # Extract features using the ResNet50 model
    feat = model.predict(np.expand_dims(img, axis=0)).ravel()
    # Save the features for this poster
    features[poster_id] = feat

In [47]:
import os
import numpy as np
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image

# Load the pre-trained ResNet50 model
model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

# Define a function to extract features from an image
def extract_features(model, img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    feat = model.predict(img).ravel()
    return feat

# Loop over all images in the "Multi_Label_dataset/images" folder
folder_path = "dataset/Multi_Label_dataset/images/"
for filename in os.listdir(folder_path):
    if filename.endswith(".jpg"):
        poster_path = os.path.join(folder_path, filename)
        poster_id = filename[:-4]
            
        # Extract features from the image
        features = extract_features(model, poster_path)
        print(features)
        
        # Add the features to the "Test_AI" column of the DataFrame
        if len(features) == 2048:
            df_movies.loc[df_movies['imdb_title_id'] == poster_id, 'Test_AI'] = len(features)
        #df_movies.loc[df_movies['imdb_title_id'] == poster_id, 'Test_AI'] = list(features)